In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel, RunnablePassthrough

from dotenv import load_dotenv
load_dotenv()



True

# Indexing

In [3]:
loader = PyPDFLoader("data/Student Handbook 2025_ENG.pdf")
docs = loader.load() 

transcript = " ".join([doc.page_content for doc in docs])

In [4]:
len(transcript.split("."))

4032

In [5]:
# text chunking

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks= splitter.create_documents([transcript])
len(chunks)

150

In [7]:
print(chunks[100].page_content)

 To equalize grading between Faculty, majors, and instructors by analyzing/releasing GPA distribution
 To help determine level of achievement in "English for Academic Excellence ” 37 
 To set a selection standard for student awards, scholarships, tuition reductions and exemptions
 To provide appropriate academic guidance during advising etc.
 To serve as a guideline for Withdrawal Advisory
(3) Grade release
Grades are posted on UNIPA in August for the spring semester and January for the fall semester. EAE level 
advancement and completion are posted approximately 1 week after the last day of  classes, and grades 
for EAE winter program are posted in late March. Guarantors can also check student grades through UNIPA. 
(4) Grade appeal
If a student suspects that a course instructor made an error in grade evaluation, or if they hold doubt about 
the ev aluation of a course after checking the learning goals, grading criteria, and grading methods


In [8]:
print(chunks[101].page_content)

the ev aluation of a course after checking the learning goals, grading criteria, and grading methods 
described in the syllabus, they may either contact the course instructor directly or submit a Grade Appeal 
Request Form through the iCLA Administrative Office to inquire to the course instructor. The deadline for 
grade appeals is 3 days from the day after the day of grade release. If a grade appeal is made through the 
iCLA Administrative Office, the course instructor must respond within 5 days from the day after the correctly 
completed form is received by the office. 
(5) Graduation assessment
Graduation assessment is the assessment conducted by the Faculty Forum to determine whether a student 
has fulfilled the requirements for graduation and will be able to obtain a degree. Students are notified of 
the results of the graduation assessment  in August for the spring semester and in February for the fall 
semester. 授業実施日 補講日
Class Day Make-up Class Day 
定期試験期間 追・再試験


In [3]:
# embedding generation

In [4]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
try:
    vector_store = FAISS.load_local("student_handbook_db", embedding, index_name="AY-2025", allow_dangerous_deserialization=True)
except Exception:
    vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)
    vector_store.save_local("student_handbook_db", index_name="AY-2025")
vector_store.index_to_docstore_id

# Retrieval

In [10]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={'k': 3})

query = "class hours"
# query = "leave"

docs = retriever.invoke(query)
for i, doc in enumerate(docs):
    print(f"Result: {i+1}")
    print(doc.page_content)
    print("\n\n\n\n\n")

Result: 1
③ Class hours
1st Period 2nd Period Lunch Break 3rd Period 4th Period 5th Period 6th Period 
9:00-10:15 10:25-11:40 11:40–12:40 12:40–13:55 14:05–15:20 15:30–16:45 16:55–18:10 
④ Credit system
At universities, students study within a carefully designed credit system. The credit system is a system in 
which students earn credits by taking courses, passing the necessary exams, and are awarded a degree 
when the number of credits earned reaches a design ated total. A credit is an indicator of how much time 
is required to acquire the contents of a course, and the standard is that 1 credit equals 45 hours of study, 
which includes not only class time but also time spent studying outside of class (preparation, review, etc.). 
The credits for each course and credits transferred from exchange partners are calculated according to 
each course’s style, educational outcome, and the study time required outside class.  
⑤ Advising system






Result: 2
typhoid fever, paratyphoid fever, 

# Augmentation

In [11]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [12]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant. 
    Answer ONLY from the provided transcript context. 
    Answer in bullet form.
    If the context is insufficient, just say you don't know.
    Transcript: {context}
    Question: {question}
    """, 
    input_variables=["context", "question"]
)

# question = "what is class hours?"
question = "what is leave policy?"
retrieved_docs = retriever.invoke(question)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

augmented_prompt = prompt.invoke({"context": context_text, "question": question})

# Generation

In [13]:
answer = llm.invoke(augmented_prompt)
print(answer.content)

- Duration of Leave of Absence: 1 or 2 semesters.
- Can be extended for up to an additional 2 semesters under special circumstances.
- Maximum total duration: 8 semesters.
- Leave of Absence does not count toward the Period of Study.
- Graduation is postponed by the number of semesters of Leave of Absence.
- Student's year level and semester remain the same during Leave of Absence.
- Leave of Absence counts towards a student’s Maximum Enrollment Period.


# using chain

In [3]:
from langchain_community.document_loaders import PyPDFLoader


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel, RunnablePassthrough

from dotenv import load_dotenv
load_dotenv()


# load PDF
file_path = "data/Student Handbook 2025_ENG.pdf"
loader = PyPDFLoader(file_path)
# loader = PDFPlumberLoader(file_path)
# loader = UnstructuredPDFLoader(file_path) # import error



docs = loader.load() 
transcript = " ".join([doc.page_content for doc in docs])

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks= splitter.create_documents([transcript])


# loading vector store
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
try:
    vector_store = FAISS.load_local("student_handbook_db", embedding, index_name="AY-2025", allow_dangerous_deserialization=True)
except Exception:
    vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)
    vector_store.save_local("student_handbook_db", index_name="AY-2025")



llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
parser = StrOutputParser()


retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={'k': 10, 'lambda_mult': 0.25})

aug_prompt = PromptTemplate(
                            template="""
                            You are a helpful assistant. 
                            Answer ONLY from the provided transcript context. 
                            Answer in bullet form.
                            If the context is insufficient, just say you don't know.
                            Transcript: {context}
                            Question: {question}
                            """, 
                            input_variables=["context", "question"]
                            )

# chains
context_chain = RunnableSequence(retriever, RunnableLambda(lambda retrieved_docs: "\n\n".join([doc.page_content for doc in retrieved_docs])))
aug_prompt_chain = RunnableSequence(aug_prompt, llm)


parallel_chain = RunnableParallel({"context": context_chain, "question": RunnablePassthrough()})

chain = RunnableSequence(parallel_chain, aug_prompt_chain, parser)


# question = "who is the author of this document?"
# question = "what is class hours?"
# question = "what is leave policy?"
# question = "explain seminars"
# question = "i do not understand the grp section"
# question = "what all courses in Data Science?"
# question = "which data science courses are offered in the Fall semester?"
# question = "which data science courses are offered in the Spring semester?"
# question = "fall semester finance courses?"
# question = "fall semester data science courses"
# question = "pre-requisite for Mathematics for Data Science"
# question = "pre-requisite for Machine Learning"
# question = "how many semesters a student need to spend to graduate?"

# question = "what is the japanese language requirement for graduation?"
# question = "how many workshops credits are necessary to graduate?"
# question = "major requirement for psychology?"
# question = "major requirement for data science?"
# question = "major requirement for political science?"
# question = "pre-requisite for Political Economy of Development"

# question = "I want to take a leave, what should I do?"

# question = "what is the minimum credits required for graduation?"


# question = "can I major in Business and minor in Data Science?"

question = "when is economic psychology offered?"

result = chain.invoke(question)
print(result)

- The transcript does not provide information on when "Economic Psychology" is offered.


# final code

In [34]:
# use this code for testing
# (chain-batch)

In [15]:
from langchain_community.document_loaders import PyPDFLoader


from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel, RunnablePassthrough

from dotenv import load_dotenv
load_dotenv()


# load PDF
file_path = "data/Student Handbook 2025_ENG.pdf"
loader = PyPDFLoader(file_path)
# loader = PDFPlumberLoader(file_path)
# loader = UnstructuredPDFLoader(file_path) # import error



docs = loader.load() 
transcript = " ".join([doc.page_content for doc in docs])

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks= splitter.create_documents([transcript])


# loading vector store
model = "OpenAI"
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
try:
    vector_store = FAISS.load_local("student_handbook_db", embedding, index_name=model, allow_dangerous_deserialization=True)
except Exception:
    vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)
    vector_store.save_local("student_handbook_db", index_name=model)



llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
parser = StrOutputParser()


retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={'k': 10, 'lambda_mult': 0.25})

aug_prompt = PromptTemplate(
                            template="""
                            You are a helpful assistant. 
                            Answer ONLY from the provided transcript context. 
                            Answer in bullet form.
                            If the context is insufficient, just say you don't know.
                            Transcript: {context}
                            Question: {question}
                            """, 
                            input_variables=["context", "question"]
                            )

# chains
context_chain = RunnableSequence(retriever, RunnableLambda(lambda retrieved_docs: "\n\n".join([doc.page_content for doc in retrieved_docs])))
aug_prompt_chain = RunnableSequence(aug_prompt, llm)


parallel_chain = RunnableParallel({"context": context_chain, "question": RunnablePassthrough()})

chain = RunnableSequence(parallel_chain, aug_prompt_chain, parser)


with open("questions.txt", "r", encoding="utf-8") as f:
    questions = [line.strip() for line in f if line.strip()]
    
answers = chain.batch(questions, config={"max_concurrency": 4})

results = dict(zip(questions, answers))
with open(f"qa_results_{model}.txt", "w", encoding="utf-8") as f:
    for q, a in results.items():
        f.write(f"Q: {q}\n")
        f.write(f"A: {a}\n")
        f.write("-"*50 + "\n\n")

In [18]:
# question = "what is the minimum credits required for graduation?"
# question = "who is the dean of the college?"
# question = "how much compensation is given in case of death on the campus?"
question = "I want to take a leave, what should I do?"
result = chain.invoke(question)
print(result)

- Obtain permission from the President.
- Ensure you are unable to study for 3 or more months.
- The duration of the Leave of Absence can be 1 or 2 semesters.
- Note that it can be extended for up to a certain period (not specified in the context).
